# P65 — Un programa de máquina para demostración de teoremas

## 1. Título y paper

**Paper:** *A Machine Program for Theorem-Proving*  
**Autoría:** Martin Davis, George Logemann, Donald Loveland  
**Año y venue:** 1962 · Communications of the ACM, 5(7), 394–397  
**Nivel:** L2 · **Motor:** `dpll`  
**Ficha completa:** [`P65_dpll`](../../papers/foundational/P65_dpll/README.md)

**Hito:** El algoritmo que sigue siendo el esqueleto de todo solucionador SAT moderno: propagar primero, ramificar solo cuando no queda deducción por hacer.

- [doi:10.1145/368273.368557](https://doi.org/10.1145/368273.368557)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: El procedimiento de Davis y Putnam (1960) era correcto pero consumía memoria de forma impracticable al eliminar variables por resolución.
2. Ejecutar una implementación mínima de la propuesta: Sustituir la eliminación por una búsqueda en profundidad con retroceso, apoyada en dos reglas que no requieren elegir: propagación de cláusulas unitarias y literales puros.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Davis y Putnam (1960), procedimiento por eliminación
- Herbrand (1930), semidecidibilidad de primer orden


## 4. Intuición

Antes de adivinar, deduce. Si una cláusula ha quedado con un solo literal, ese literal no es una opción: es una obligación. Aplicar todas las obligaciones antes de tomar cualquier decisión es lo que separa un solucionador viable de una tabla de verdad.


## 5. Concepto mínimo

```text
DPLL(F, asignación):
    si F está vacía        → SATISFACIBLE
    si hay cláusula vacía  → conflicto, retroceder
    si hay cláusula unitaria (L)  → asignar L, repetir      ← deducción
    si hay literal puro L         → asignar L, repetir      ← deducción
    elegir variable v y probar v=1, luego v=0               ← decisión
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('dpll', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántos nodos visitará DPLL frente a las 32 filas de la tabla de verdad?
2. ¿Cuántos de esos pasos son deducción y cuántos decisión?
3. ¿Cuántas asignaciones satisfacen la fórmula?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('dpll', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('dpll', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

DPLL visita **5 nodos** frente a las 32 filas de la tabla: un factor de 6,4× con solo cinco variables. De esos pasos, 3 son propagaciones unitarias, es decir deducción pura. Solo 3 de las 32 asignaciones satisfacen la fórmula.


## 10. Comentario pedagógico

Este algoritmo tiene sesenta años y sigue siendo el esqueleto de los solucionadores actuales, que resuelven instancias de millones de cláusulas. Lo que se añadió después —aprendizaje de cláusulas, reinicios, heurísticas de actividad— se monta encima de este bucle, no lo sustituye.


## 11. Error o anti-patrón deliberado

Anti-patrón: tratar la propagación unitaria como una heurística que se puede deshacer.


In [ ]:
print('Una clausula unitaria (L) no deja eleccion: L TIENE que ser cierto.')
print('Eso es una deduccion, no una apuesta, y no se retrocede sobre ella.')
print('Confundir deduccion con decision es el error clasico al implementar DPLL.')

## 12. Corrección

La separación correcta entre lo que se deduce y lo que se decide:


In [ ]:
r = run_paper_lab('dpll', seed=7)['result']
print('nodos totales        :', r['nodos_dpll'])
print('propagacion unitaria :', r['propagaciones_unitarias'], '<- deduccion')
print('literales puros      :', r['literales_puros'], '<- deduccion')
print('tabla completa       :', r['asignaciones_tabla_completa'], 'filas')

## 13. Desafío guiado

Comprueba en la salida cuántas asignaciones satisfacen la fórmula y compáralo con lo que costaría encontrarlas por sondeo aleatorio si hubiera 50 variables en vez de 5.


In [ ]:
r = run_paper_lab('dpll', seed=3)['result']
show(r)

## 14. Desafío autónomo

Codifica un sudoku de 4×4 en forma normal conjuntiva y resuélvelo con este motor. Cuenta cuántas variables y cláusulas necesitas, y cuántos nodos hacen falta con y sin propagación.


## 15. Evidencia de aprendizaje

Guarda el conteo de nodos frente a la tabla completa y tu distinción entre los pasos que son deducción y los que son decisión.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P65_dpll/README.md) · evaluación formal: [`assessments/papers/P65_dpll.md`](../../assessments/papers/P65_dpll.md)


## 16. Cierre

Ya se decide en lógica proposicional. El salto siguiente es poder hablar de objetos y relaciones, y para eso hace falta una regla de inferencia que sepa igualar términos.


## 17. Conexión con el siguiente hito

- P66

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
